In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "shunk031/HaluEval",
    "qa"
)

print(dataset)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['knowledge', 'question', 'right_answer', 'hallucinated_answer'],
        num_rows: 10000
    })
})


In [3]:
print(dataset["train"].column_names)
print(dataset["train"][0])

['knowledge', 'question', 'right_answer', 'hallucinated_answer']
{'knowledge': "Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.", 'question': "Which magazine was started first Arthur's Magazine or First for Women?", 'right_answer': "Arthur's Magazine", 'hallucinated_answer': 'First for Women was started first.'}


In [4]:
halueval_df = dataset["train"].to_pandas()

print("Shape:", halueval_df.shape)
print("Columns:", halueval_df.columns.tolist())

Shape: (10000, 4)
Columns: ['knowledge', 'question', 'right_answer', 'hallucinated_answer']


In [5]:
print("QUESTION:")
print(halueval_df.iloc[0]["question"])

print("\nKNOWLEDGE:")
print(halueval_df.iloc[0]["knowledge"])

print("\nRIGHT ANSWER:")
print(halueval_df.iloc[0]["right_answer"])

print("\nHALLUCINATED ANSWER:")
print(halueval_df.iloc[0]["hallucinated_answer"])

QUESTION:
Which magazine was started first Arthur's Magazine or First for Women?

KNOWLEDGE:
Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.

RIGHT ANSWER:
Arthur's Magazine

HALLUCINATED ANSWER:
First for Women was started first.


In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "shunk031/HaluEval",
    "qa"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['knowledge', 'question', 'right_answer', 'hallucinated_answer'],
        num_rows: 10000
    })
})


In [7]:
print(dataset["train"].column_names)
print(dataset["train"][0])

['knowledge', 'question', 'right_answer', 'hallucinated_answer']
{'knowledge': "Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.", 'question': "Which magazine was started first Arthur's Magazine or First for Women?", 'right_answer': "Arthur's Magazine", 'hallucinated_answer': 'First for Women was started first.'}


In [8]:
import importlib
import src.data.preprocessing as preprocessing

importlib.reload(preprocessing)

print("convert_halueval" in dir(preprocessing))

True


In [9]:
convert_halueval = preprocessing.convert_halueval

In [10]:
halueval_processed = convert_halueval(
    halueval_df
)

print("Shape:", halueval_processed.shape)
print(
    "Columns:",
    halueval_processed.columns.tolist()
)

Shape: (20000, 8)
Columns: ['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category']


In [11]:
print(
    halueval_processed.head(2).to_string(
        index=False
    )
)

question_id   candidate_id source_dataset                                                               question                             answer                                                                                                                                                                                          context  label question_category
   HE_00000 HE_00000_R_000       halueval Which magazine was started first Arthur's Magazine or First for Women?                  Arthur's Magazine Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.      0                QA
   HE_00000 HE_00000_H_000       halueval Which magazine was started first Arthur's Magazine or First for Women? First for Women was started first. Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First fo

In [12]:
print(
    halueval_processed["label"].value_counts()
)

label
0    10000
1    10000
Name: count, dtype: int64


In [13]:
first_question = halueval_processed.iloc[0]["question"]

print(
    halueval_processed[
        halueval_processed["question"] == first_question
    ][
        [
            "question_id",
            "candidate_id",
            "question",
            "answer",
            "context",
            "label"
        ]
    ].to_string(index=False)
)

question_id   candidate_id                                                               question                             answer                                                                                                                                                                                          context  label
   HE_00000 HE_00000_R_000 Which magazine was started first Arthur's Magazine or First for Women?                  Arthur's Magazine Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.      0
   HE_00000 HE_00000_H_000 Which magazine was started first Arthur's Magazine or First for Women? First for Women was started first. Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.      1


In [14]:
print("Rows:", len(halueval_processed))

print(
    "Unique question IDs:",
    halueval_processed["question_id"].nunique()
)

print(
    "Unique candidate IDs:",
    halueval_processed["candidate_id"].nunique()
)

print(
    "Missing values:"
)

print(
    halueval_processed.isnull().sum()
)

Rows: 20000
Unique question IDs: 10000
Unique candidate IDs: 20000
Missing values:
question_id          0
candidate_id         0
source_dataset       0
question             0
answer               0
context              0
label                0
question_category    0
dtype: int64


In [15]:
print(
    "Empty questions:",
    (
        halueval_processed["question"]
        .fillna("")
        .str.strip()
        .eq("")
    ).sum()
)

print(
    "Empty answers:",
    (
        halueval_processed["answer"]
        .fillna("")
        .str.strip()
        .eq("")
    ).sum()
)

print(
    "Empty contexts:",
    (
        halueval_processed["context"]
        .fillna("")
        .str.strip()
        .eq("")
    ).sum()
)

Empty questions: 0
Empty answers: 0
Empty contexts: 0


In [16]:
print(
    "Duplicate Q&A pairs:",
    halueval_processed.duplicated(
        subset=["question", "answer"]
    ).sum()
)

Duplicate Q&A pairs: 0


In [17]:
halueval_processed = convert_halueval(halueval_df)

print("Shape:", halueval_processed.shape)
print("Columns:", halueval_processed.columns.tolist())
print(halueval_processed["label"].value_counts())

Shape: (20000, 8)
Columns: ['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category']
label
0    10000
1    10000
Name: count, dtype: int64


In [18]:
halueval_cleaned = halueval_processed.copy()

In [19]:
halueval_cleaned.to_parquet(
    "../data/processed/halueval_cleaned.parquet",
    index=False
)

In [20]:
import os

path = "../data/processed/halueval_cleaned.parquet"

print("File exists:", os.path.exists(path))
print("Rows:", len(halueval_cleaned))

File exists: True
Rows: 20000


In [21]:
from sklearn.model_selection import train_test_split
import numpy as np

question_ids = np.array(
    halueval_cleaned["question_id"].unique()
)

print("Total unique questions:", len(question_ids))

train_ids, temp_ids = train_test_split(
    question_ids,
    test_size=0.30,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

print("Train questions:", len(train_ids))
print("Validation questions:", len(val_ids))
print("Test questions:", len(test_ids))

Total unique questions: 10000
Train questions: 7000
Validation questions: 1500
Test questions: 1500


In [22]:
halueval_train = halueval_cleaned[
    halueval_cleaned["question_id"].isin(train_ids)
].copy()

halueval_val = halueval_cleaned[
    halueval_cleaned["question_id"].isin(val_ids)
].copy()

halueval_test = halueval_cleaned[
    halueval_cleaned["question_id"].isin(test_ids)
].copy()

In [23]:
print("Train rows:", len(halueval_train))
print("Validation rows:", len(halueval_val))
print("Test rows:", len(halueval_test))

print(
    "\nTrain questions:",
    halueval_train["question_id"].nunique()
)

print(
    "Validation questions:",
    halueval_val["question_id"].nunique()
)

print(
    "Test questions:",
    halueval_test["question_id"].nunique()
)

Train rows: 14000
Validation rows: 3000
Test rows: 3000

Train questions: 7000
Validation questions: 1500
Test questions: 1500


In [25]:
train_questions = set(
    halueval_train["question_id"]
)

val_questions = set(
    halueval_val["question_id"]
)

test_questions = set(
    halueval_test["question_id"]
)

print(
    "Train ∩ Validation:",
    len(train_questions & val_questions)
)

print(
    "Train ∩ Test:",
    len(train_questions & test_questions)
)

print(
    "Validation ∩ Test:",
    len(val_questions & test_questions)
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [26]:
halueval_train.to_parquet(
    "../data/processed/halueval_train.parquet",
    index=False
)

halueval_val.to_parquet(
    "../data/processed/halueval_validation.parquet",
    index=False
)

halueval_test.to_parquet(
    "../data/processed/halueval_test.parquet",
    index=False
)